In [186]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
from sentence_transformers import SentenceTransformer
import joblib


In [187]:
df=pd.read_csv(r"P:\AI-Powered Document Classification and Intelligent Indexing\data\csv_dataset\extracted_text.csv")
encoder=LabelEncoder()
df['Category_Encoded'] = encoder.fit_transform(df['Category'])
df.head()

,Category,Content,Category_Encoded
0,finance,management s discussion and analysis md a quar...,1
1,finance,internal audit compliance memo sox 404 testing...,1
2,finance,commercial credit underwriting memo senior sec...,1
3,finance,capex budget variance report q2 capex budget v...,1
4,finance,investment committee brief lbo project spark a...,1


In [188]:
embedder=SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

In [189]:
contents=df['Content']
averaged_vector=[]
for content in contents:
    vectors_=[]
    sum=0
    sentences=content.split(" ")
    chunks=[]
    for words in sentences:

        if len(chunks)==200:
            chunk_string = " ".join(chunks)
            vector = embedder.encode(chunk_string)
            vectors_.append(vector)
            chunks=[]
        chunks.append(words)
    if len(chunks) > 0:
        chunk_string = " ".join(chunks)
        vector = embedder.encode(chunk_string)
        vectors_.append(vector)
    if len(vectors_)>0:
        for vector in vectors_:
            sum+=vector
        average=sum/len(vectors_)
        averaged_vector.append(average)
print(averaged_vector)



[array([-2.31561288e-02,  1.57301016e-02, -9.75099951e-03,  2.16912050e-02,
        6.15881160e-02,  1.23385806e-03, -8.86200555e-03,  2.73497775e-02,
       -9.11912918e-02, -2.47315783e-02, -6.44888915e-03,  4.46607843e-02,
        1.41864605e-02,  6.43030480e-02,  2.40822453e-02,  6.63764849e-02,
        1.06576807e-03, -1.84884109e-02, -3.56445760e-02,  1.10712778e-02,
       -3.14105861e-03,  1.92872807e-02, -2.53360197e-02, -1.91238243e-02,
        1.78305828e-03,  2.67930813e-02, -4.02429402e-02, -1.66880414e-02,
       -2.45861313e-03, -6.91423267e-02,  1.32993823e-02, -1.57480277e-02,
        2.62135528e-02,  1.11712283e-03,  2.08782376e-06, -4.55285385e-02,
       -4.35309596e-02,  1.93468686e-02, -1.63328387e-02,  3.43301669e-02,
       -5.60572743e-03, -3.52626760e-03,  2.00864840e-02,  6.61057420e-05,
        3.78805841e-03, -3.72016653e-02,  1.38111254e-02,  3.58697288e-02,
       -2.73175798e-02,  3.34937125e-03,  2.00946294e-02, -1.05263973e-02,
       -2.92161331e-02, 

In [190]:
x_train,x_test,y_train,y_test=train_test_split(averaged_vector,df['Category_Encoded'],test_size=0.3,random_state=42)

In [191]:
model=LogisticRegression(C=1.0,class_weight='balanced', max_iter=1000, random_state=42)
model.fit(x_train,y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:

In [192]:
y_pred=model.predict(x_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.75      1.00      0.86        12
           1       0.92      0.85      0.88        13
           2       0.89      1.00      0.94         8
           3       0.86      0.60      0.71        10
           4       1.00      0.83      0.91         6

    accuracy                           0.86        49
   macro avg       0.88      0.86      0.86        49
weighted avg       0.87      0.86      0.85        49



In [193]:
new_text = ["The company reported a net revenue increase in Q3."]
new_emb = embedder.encode(new_text)

predicted_category = encoder.inverse_transform(model.predict(new_emb))[0]
print(f"Predicted: {predicted_category.upper()}")

Predicted: FINANCE


In [195]:
joblib.dump(model, '../models/Logistic_Model.joblib')
joblib.dump(encoder, '../models/Logistic_Encoder.joblib')
embedder.save('sentence_transformer_model')